[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MaxiRuess/DeepLearning_101/blob/main/notebooks/06_Kernels/04_LayerNorm.ipynb)

# LayerNorm — Fused Normalization Kernels

This notebook implements both **LayerNorm** and **RMSNorm** as fused Triton kernels. Normalization is the **most practical** custom kernel — it's what AI labs ship first, because every transformer block has 1-2 normalization calls and the fusion benefit is large.

| | Matrix Multiply (previous) | LayerNorm (this notebook) |
|---|---|---|
| Bottleneck | Compute-bound (FLOPs) | **Memory-bound (HBM bandwidth)** |
| Grid | 2D (one program per output tile) | **1D (one program per row, same as softmax)** |
| Key optimization | Tiling (reuse data in SRAM) | **Fusion (1 pass vs 4+)** |
| Triton advantage | Approaches cuBLAS TFLOPS | **Eliminates 3+ extra HBM round-trips** |
| New Triton concepts | `tl.dot`, 2D grid, K-loop | **Learnable parameters (gamma, beta), two reductions** |

> This kernel uses the same fusion principle as [Softmax](./02_Softmax.ipynb) — multiple HBM passes collapsed into one SRAM pass — but adds learnable weight parameters and a second reduction (variance).

## Setup

In [ ]:
import sys, os

# In Colab, clone the repo so local imports (kernels/, src/) work
if "google.colab" in str(get_ipython()):
    if not os.path.exists("/content/DeepLearning_101"):
        !git clone --depth 1 https://github.com/MaxiRuess/DeepLearning_101.git /content/DeepLearning_101
    os.chdir("/content/DeepLearning_101/notebooks/06_Kernels")
    sys.path.insert(0, "/content/DeepLearning_101")
else:
    sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..', '..')))

In [2]:
# Detect runtime environment
import torch

IN_COLAB = "google.colab" in str(get_ipython()) if hasattr(__builtins__, '__IPYTHON__') else False
HAS_CUDA = torch.cuda.is_available()

if IN_COLAB:
    %pip install -q triton
    print(f"Running in Colab with GPU: {torch.cuda.get_device_name(0)}")
elif HAS_CUDA:
    print(f"Running locally with GPU: {torch.cuda.get_device_name(0)}")
else:
    print("No GPU detected — will use Modal for remote GPU execution")
    print("Make sure you have Modal configured: pip install modal && modal token set")

No GPU detected — will use Modal for remote GPU execution
Make sure you have Modal configured: pip install modal && modal token set


## Why LayerNorm Matters

Every transformer block calls LayerNorm (or RMSNorm) 1-2 times. PyTorch's `nn.LayerNorm` does 4+ separate operations, each reading the full row from slow HBM:

```
PyTorch nn.LayerNorm (4+ passes over HBM):

  Pass 1: mean       Pass 2: variance     Pass 3: normalize    Pass 4: scale+shift
  ┌─────────┐        ┌─────────┐          ┌─────────┐          ┌─────────┐
  │ read x  │→ mean  │ read x  │→ var     │ read x  │→ norm   │ read norm│→ output
  └─────────┘        │ read mean│          │ read mean│         │ read γ,β │
                      └─────────┘          │ read var │         └─────────┘
                                            └─────────┘
  4+ reads of x (or intermediates) from slow HBM


Triton fused LayerNorm (1 pass):
  ┌─────────┐
  │ read x  │→ SRAM: mean → var → normalize → scale+shift → write
  │ read γ,β│   (all reductions + element-wise in fast on-chip memory)
  └─────────┘
  1 read of x + 1 read of weights + 1 write — that's it!
```

Same fusion principle as [Softmax](./02_Softmax.ipynb) (3 passes → 1), but even more passes eliminated.

## The Math

**LayerNorm** (Ba et al., 2016):

$\mu = \frac{1}{D}\sum_{i=1}^{D} x_i \qquad \sigma^2 = \frac{1}{D}\sum_{i=1}^{D} (x_i - \mu)^2$

$\hat{x}_i = \frac{x_i - \mu}{\sqrt{\sigma^2 + \epsilon}} \qquad y_i = \gamma \hat{x}_i + \beta$

Where $\gamma$ (scale) and $\beta$ (shift) are **learnable parameters** of shape $(D,)$.

**RMSNorm** (Zhang & Sennrich, 2019) — used by LLaMA, Mistral, Gemma, Qwen:

$\text{RMS}(x) = \sqrt{\frac{1}{D}\sum_{i=1}^{D} x_i^2 + \epsilon} \qquad y_i = \gamma \cdot \frac{x_i}{\text{RMS}(x)}$

Key differences: **no mean subtraction**, **no beta**. Only 1 reduction instead of 2.

## Naive Python LayerNorm (4-Pass)

This is conceptually what PyTorch does — four separate operations, each touching HBM:

In [3]:
def naive_layer_norm(x: torch.Tensor, weight: torch.Tensor, bias: torch.Tensor,
                     eps: float = 1e-5) -> torch.Tensor:
    """LayerNorm the PyTorch way — 4 separate passes over the data."""
    # Pass 1: Read x from HBM, compute mean, write mean to HBM
    mean = x.mean(dim=-1, keepdim=True)

    # Pass 2: Read x and mean from HBM, compute variance, write to HBM
    var = ((x - mean) ** 2).mean(dim=-1, keepdim=True)

    # Pass 3: Read x, mean, var from HBM, normalize, write to HBM
    x_norm = (x - mean) / torch.sqrt(var + eps)

    # Pass 4: Read x_norm, weight, bias from HBM, scale+shift, write to HBM
    return weight * x_norm + bias

## The Triton LayerNorm Kernel — Explained

Our kernel fuses all four passes into one. Key differences from the [Softmax kernel](./02_Softmax.ipynb):

- **Two reductions** — mean AND variance (softmax only needed max and sum)
- **Learnable parameters** — first kernel that loads external weights (`gamma`, `beta`) from separate pointers
- **`0.0` padding** (not `-inf`) — zeros don't affect mean/variance when we divide by `n_cols`. Contrast with softmax where `-inf` ensures `exp(-inf) = 0`
- **Same grid as softmax** — 1D, one program per row, `BLOCK_SIZE >= n_cols`

The full code lives in `kernels/layer_norm.py`.

In [ ]:
from kernels.layer_norm import layer_norm_kernel, layer_norm, rms_norm_kernel, rms_norm
from pathlib import Path
print(Path("kernels/layer_norm.py").read_text())

## RMSNorm — The Modern Alternative

Virtually all modern open-source LLMs use RMSNorm instead of LayerNorm:

| Model | Normalization |
|---|---|
| BERT, GPT-2 | LayerNorm |
| LLaMA 1/2/3 | **RMSNorm** |
| Mistral, Mixtral | **RMSNorm** |
| Gemma | **RMSNorm** |
| Qwen 2 | **RMSNorm** |
| DeepSeek-V3 | **RMSNorm** |

Why? The mean subtraction in LayerNorm provides minimal benefit for the extra compute cost. RMSNorm is simpler (1 reduction vs 2) and empirically performs just as well.

## How the Grid Works

Same structure as softmax — one program per row:

```
Input matrix (N x D):
┌────────────────────────────────────────┐
│ Row 0: [x₀₀, x₀₁, ..., x₀d]         │ → Program 0 (mean, var, norm, scale+shift)
│ Row 1: [x₁₀, x₁₁, ..., x₁d]         │ → Program 1
│  ...                                    │    ...
│ Row N: [xₙ₀, xₙ₁, ..., xₙd]         │ → Program N
└────────────────────────────────────────┘
              +                  +
Weight γ:  [γ₀, γ₁, ..., γd]   ← shared across all rows
Bias   β:  [β₀, β₁, ..., βd]   ← shared across all rows (LayerNorm only)

Grid = (n_rows,)  ← same as softmax
```

| | Vector Add | Softmax | Matmul | **LayerNorm** |
|---|---|---|---|---|
| Grid | 1D | 1D | 2D | **1D** |
| Each program | Chunk of elements | One full row | Output tile | **One full row** |
| Reductions | 0 | 2 (max, sum) | 0 | **2 (mean, var)** |
| External weights | No | No | No | **Yes (gamma, beta)** |

## Run on GPU

Triton requires an NVIDIA GPU. This notebook supports two execution modes:
- **Colab / Local CUDA** — runs directly on the available GPU
- **Modal** — runs on a remote T4 GPU (for Mac / no-GPU machines)

In [4]:
import time

def benchmark_layernorm():
    """Run correctness test + benchmark. Works on any CUDA device."""
    import triton
    import triton.language as tl

    # --- Inline LayerNorm kernel ---
    @triton.jit
    def _layer_norm_kernel(
        input_ptr, output_ptr, weight_ptr, bias_ptr,
        n_cols, input_row_stride, output_row_stride, eps,
        BLOCK_SIZE: tl.constexpr,
    ):
        row_idx = tl.program_id(axis=0)
        row_start_input = input_ptr + row_idx * input_row_stride
        row_start_output = output_ptr + row_idx * output_row_stride
        col_offsets = tl.arange(0, BLOCK_SIZE)
        mask = col_offsets < n_cols
        x = tl.load(row_start_input + col_offsets, mask=mask, other=0.0)
        mean = tl.sum(x, axis=0) / n_cols
        x_centered = x - mean
        var = tl.sum(x_centered * x_centered, axis=0) / n_cols
        x_norm = x_centered / tl.sqrt(var + eps)
        weight = tl.load(weight_ptr + col_offsets, mask=mask, other=1.0)
        bias = tl.load(bias_ptr + col_offsets, mask=mask, other=0.0)
        y = weight * x_norm + bias
        tl.store(row_start_output + col_offsets, y, mask=mask)

    # --- Inline RMSNorm kernel ---
    @triton.jit
    def _rms_norm_kernel(
        input_ptr, output_ptr, weight_ptr,
        n_cols, input_row_stride, output_row_stride, eps,
        BLOCK_SIZE: tl.constexpr,
    ):
        row_idx = tl.program_id(axis=0)
        row_start_input = input_ptr + row_idx * input_row_stride
        row_start_output = output_ptr + row_idx * output_row_stride
        col_offsets = tl.arange(0, BLOCK_SIZE)
        mask = col_offsets < n_cols
        x = tl.load(row_start_input + col_offsets, mask=mask, other=0.0)
        rms = tl.sqrt(tl.sum(x * x, axis=0) / n_cols + eps)
        x_norm = x / rms
        weight = tl.load(weight_ptr + col_offsets, mask=mask, other=1.0)
        y = weight * x_norm
        tl.store(row_start_output + col_offsets, y, mask=mask)

    def triton_layer_norm(x, weight, bias, eps=1e-5):
        n_rows, n_cols = x.shape
        output = torch.empty_like(x)
        BLOCK_SIZE = triton.next_power_of_2(n_cols)
        _layer_norm_kernel[(n_rows,)](
            x, output, weight, bias, n_cols,
            x.stride(0), output.stride(0), eps, BLOCK_SIZE=BLOCK_SIZE,
        )
        return output

    def triton_rms_norm(x, weight, eps=1e-5):
        n_rows, n_cols = x.shape
        output = torch.empty_like(x)
        BLOCK_SIZE = triton.next_power_of_2(n_cols)
        _rms_norm_kernel[(n_rows,)](
            x, output, weight, n_cols,
            x.stride(0), output.stride(0), eps, BLOCK_SIZE=BLOCK_SIZE,
        )
        return output

    # --- Correctness tests ---
    torch.manual_seed(0)
    N, D = 128, 512
    x = torch.randn(N, D, device="cuda")
    weight = torch.randn(D, device="cuda")
    bias = torch.randn(D, device="cuda")
    eps = 1e-5

    # LayerNorm
    out_triton_ln = triton_layer_norm(x, weight, bias, eps)
    out_torch_ln = torch.layer_norm(x, [D], weight, bias, eps)
    max_diff_ln = (out_triton_ln - out_torch_ln).abs().max().item()
    match_ln = torch.allclose(out_triton_ln, out_torch_ln, atol=1e-5)
    print(f"LayerNorm max diff: {max_diff_ln:.2e}, match: {match_ln}")

    # RMSNorm
    rms_ref = torch.sqrt(x.pow(2).mean(dim=-1, keepdim=True) + eps)
    out_ref_rms = weight * (x / rms_ref)
    out_triton_rms = triton_rms_norm(x, weight, eps)
    max_diff_rms = (out_triton_rms - out_ref_rms).abs().max().item()
    match_rms = torch.allclose(out_triton_rms, out_ref_rms, atol=1e-5)
    print(f"RMSNorm max diff: {max_diff_rms:.2e}, match: {match_rms}")

    # --- Benchmark: vary D with fixed N=256 ---
    n_rows = 256
    col_sizes = [64, 128, 256, 512, 1024, 2048, 4096]
    naive_times = []
    torch_times = []
    triton_ln_times = []
    triton_rms_times = []

    for D in col_sizes:
        x = torch.randn(n_rows, D, device="cuda")
        w = torch.ones(D, device="cuda")
        b = torch.zeros(D, device="cuda")

        # Warmup
        for _ in range(10):
            naive_layer_norm(x, w, b, eps)
            torch.layer_norm(x, [D], w, b, eps)
            triton_layer_norm(x, w, b, eps)
            triton_rms_norm(x, w, eps)
        torch.cuda.synchronize()

        # Naive (4-pass)
        start = time.perf_counter()
        for _ in range(100):
            naive_layer_norm(x, w, b, eps)
        torch.cuda.synchronize()
        naive_times.append((time.perf_counter() - start) / 100)

        # PyTorch built-in
        start = time.perf_counter()
        for _ in range(100):
            torch.layer_norm(x, [D], w, b, eps)
        torch.cuda.synchronize()
        torch_times.append((time.perf_counter() - start) / 100)

        # Triton LayerNorm
        start = time.perf_counter()
        for _ in range(100):
            triton_layer_norm(x, w, b, eps)
        torch.cuda.synchronize()
        triton_ln_times.append((time.perf_counter() - start) / 100)

        # Triton RMSNorm
        start = time.perf_counter()
        for _ in range(100):
            triton_rms_norm(x, w, eps)
        torch.cuda.synchronize()
        triton_rms_times.append((time.perf_counter() - start) / 100)

    return {
        "match_ln": match_ln,
        "match_rms": match_rms,
        "max_diff_ln": max_diff_ln,
        "max_diff_rms": max_diff_rms,
        "col_sizes": col_sizes,
        "naive_us": [t * 1e6 for t in naive_times],
        "torch_us": [t * 1e6 for t in torch_times],
        "triton_ln_us": [t * 1e6 for t in triton_ln_times],
        "triton_rms_us": [t * 1e6 for t in triton_rms_times],
    }

In [ ]:
if HAS_CUDA:
    # --- Direct GPU execution (Colab or local CUDA) ---
    results = benchmark_layernorm()
else:
    # --- Modal remote execution (no local GPU) ---
    # Note: @triton.jit kernels can't be serialized by Modal (source code breaks
    # after pickle), so we must define everything inline in the Modal function.
    import modal

    app = modal.App("triton-layernorm")
    image = modal.Image.debian_slim(python_version="3.12").pip_install("torch", "triton")

    @app.function(image=image, gpu="T4")
    def run_remote():
        import torch
        import triton
        import triton.language as tl
        import time

        @triton.jit
        def _layer_norm_kernel(
            input_ptr, output_ptr, weight_ptr, bias_ptr,
            n_cols, input_row_stride, output_row_stride, eps,
            BLOCK_SIZE: tl.constexpr,
        ):
            row_idx = tl.program_id(axis=0)
            row_start_input = input_ptr + row_idx * input_row_stride
            row_start_output = output_ptr + row_idx * output_row_stride
            col_offsets = tl.arange(0, BLOCK_SIZE)
            mask = col_offsets < n_cols
            x = tl.load(row_start_input + col_offsets, mask=mask, other=0.0)
            mean = tl.sum(x, axis=0) / n_cols
            x_centered = x - mean
            var = tl.sum(x_centered * x_centered, axis=0) / n_cols
            x_norm = x_centered / tl.sqrt(var + eps)
            weight = tl.load(weight_ptr + col_offsets, mask=mask, other=1.0)
            bias = tl.load(bias_ptr + col_offsets, mask=mask, other=0.0)
            y = weight * x_norm + bias
            tl.store(row_start_output + col_offsets, y, mask=mask)

        @triton.jit
        def _rms_norm_kernel(
            input_ptr, output_ptr, weight_ptr,
            n_cols, input_row_stride, output_row_stride, eps,
            BLOCK_SIZE: tl.constexpr,
        ):
            row_idx = tl.program_id(axis=0)
            row_start_input = input_ptr + row_idx * input_row_stride
            row_start_output = output_ptr + row_idx * output_row_stride
            col_offsets = tl.arange(0, BLOCK_SIZE)
            mask = col_offsets < n_cols
            x = tl.load(row_start_input + col_offsets, mask=mask, other=0.0)
            rms = tl.sqrt(tl.sum(x * x, axis=0) / n_cols + eps)
            x_norm = x / rms
            weight = tl.load(weight_ptr + col_offsets, mask=mask, other=1.0)
            y = weight * x_norm
            tl.store(row_start_output + col_offsets, y, mask=mask)

        def triton_layer_norm(x, weight, bias, eps=1e-5):
            n_rows, n_cols = x.shape
            output = torch.empty_like(x)
            BLOCK_SIZE = triton.next_power_of_2(n_cols)
            _layer_norm_kernel[(n_rows,)](
                x, output, weight, bias, n_cols,
                x.stride(0), output.stride(0), eps, BLOCK_SIZE=BLOCK_SIZE,
            )
            return output

        def triton_rms_norm(x, weight, eps=1e-5):
            n_rows, n_cols = x.shape
            output = torch.empty_like(x)
            BLOCK_SIZE = triton.next_power_of_2(n_cols)
            _rms_norm_kernel[(n_rows,)](
                x, output, weight, n_cols,
                x.stride(0), output.stride(0), eps, BLOCK_SIZE=BLOCK_SIZE,
            )
            return output

        def naive_layer_norm(x, weight, bias, eps=1e-5):
            mean = x.mean(dim=-1, keepdim=True)
            var = ((x - mean) ** 2).mean(dim=-1, keepdim=True)
            x_norm = (x - mean) / torch.sqrt(var + eps)
            return weight * x_norm + bias

        # Correctness
        torch.manual_seed(0)
        N, D = 128, 512
        x = torch.randn(N, D, device="cuda")
        weight = torch.randn(D, device="cuda")
        bias = torch.randn(D, device="cuda")
        eps = 1e-5

        out_triton_ln = triton_layer_norm(x, weight, bias, eps)
        out_torch_ln = torch.layer_norm(x, [D], weight, bias, eps)
        max_diff_ln = (out_triton_ln - out_torch_ln).abs().max().item()
        match_ln = torch.allclose(out_triton_ln, out_torch_ln, atol=1e-5)

        rms_ref = torch.sqrt(x.pow(2).mean(dim=-1, keepdim=True) + eps)
        out_ref_rms = weight * (x / rms_ref)
        out_triton_rms = triton_rms_norm(x, weight, eps)
        max_diff_rms = (out_triton_rms - out_ref_rms).abs().max().item()
        match_rms = torch.allclose(out_triton_rms, out_ref_rms, atol=1e-5)

        print(f"LayerNorm max diff: {max_diff_ln:.2e}, match: {match_ln}")
        print(f"RMSNorm max diff: {max_diff_rms:.2e}, match: {match_rms}")

        # Benchmark
        n_rows = 256
        col_sizes = [64, 128, 256, 512, 1024, 2048, 4096]
        naive_times, torch_times, triton_ln_times, triton_rms_times = [], [], [], []

        for D in col_sizes:
            x = torch.randn(n_rows, D, device="cuda")
            w = torch.ones(D, device="cuda")
            b = torch.zeros(D, device="cuda")

            for _ in range(10):
                naive_layer_norm(x, w, b, eps)
                torch.layer_norm(x, [D], w, b, eps)
                triton_layer_norm(x, w, b, eps)
                triton_rms_norm(x, w, eps)
            torch.cuda.synchronize()

            for fn, times_list in [
                (lambda: naive_layer_norm(x, w, b, eps), naive_times),
                (lambda: torch.layer_norm(x, [D], w, b, eps), torch_times),
                (lambda: triton_layer_norm(x, w, b, eps), triton_ln_times),
                (lambda: triton_rms_norm(x, w, eps), triton_rms_times),
            ]:
                start = time.perf_counter()
                for _ in range(100):
                    fn()
                torch.cuda.synchronize()
                times_list.append((time.perf_counter() - start) / 100)

        return {
            "match_ln": match_ln, "match_rms": match_rms,
            "max_diff_ln": max_diff_ln, "max_diff_rms": max_diff_rms,
            "col_sizes": col_sizes,
            "naive_us": [t * 1e6 for t in naive_times],
            "torch_us": [t * 1e6 for t in torch_times],
            "triton_ln_us": [t * 1e6 for t in triton_ln_times],
            "triton_rms_us": [t * 1e6 for t in triton_rms_times],
        }

    with app.run():
        results = run_remote.remote()

In [ ]:
print(f"LayerNorm — Correctness: {'PASS' if results['match_ln'] else 'FAIL'}, Max diff: {results['max_diff_ln']:.2e}")
print(f"RMSNorm  — Correctness: {'PASS' if results['match_rms'] else 'FAIL'}, Max diff: {results['max_diff_rms']:.2e}")

## Benchmark Results

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sizes = results["col_sizes"]

# --- Plot 1: Execution time ---
ax = axes[0]
ax.plot(sizes, results["naive_us"], "^-.", label="Naive (4-pass)", linewidth=2, color="#e74c3c")
ax.plot(sizes, results["torch_us"], "s--", label="torch.layer_norm", linewidth=2, color="#3498db")
ax.plot(sizes, results["triton_ln_us"], "o-", label="Triton LayerNorm", linewidth=2, color="#2ecc71")
ax.plot(sizes, results["triton_rms_us"], "d-", label="Triton RMSNorm", linewidth=2, color="#9b59b6")
ax.set_xscale("log", base=2)
ax.set_xlabel("Feature dimension (D)")
ax.set_ylabel("Time (microseconds)")
ax.set_title("Normalization: Execution Time vs Feature Dim")
ax.legend()
ax.grid(True, alpha=0.3)

# --- Plot 2: Speedup ---
ax = axes[1]
speedup_ln_vs_torch = [t / tr for t, tr in zip(results["torch_us"], results["triton_ln_us"])]
speedup_rms_vs_torch = [t / tr for t, tr in zip(results["torch_us"], results["triton_rms_us"])]
speedup_ln_vs_naive = [n / tr for n, tr in zip(results["naive_us"], results["triton_ln_us"])]
x_pos = np.arange(len(sizes))
width = 0.25
ax.bar(x_pos - width, speedup_ln_vs_naive, width, label="Triton LN vs naive", color="#e74c3c")
ax.bar(x_pos, speedup_ln_vs_torch, width, label="Triton LN vs torch", color="#3498db")
ax.bar(x_pos + width, speedup_rms_vs_torch, width, label="Triton RMS vs torch", color="#9b59b6")
ax.axhline(y=1.0, color="gray", linestyle="--", alpha=0.5)
ax.set_xticks(x_pos)
ax.set_xticklabels([str(s) for s in sizes])
ax.set_xlabel("Feature dimension (D)")
ax.set_ylabel("Speedup (higher = Triton wins)")
ax.set_title("Triton Speedup over PyTorch")
ax.legend()
ax.grid(True, alpha=0.3, axis="y")

plt.tight_layout()
plt.show()

## LayerNorm vs RMSNorm

| | LayerNorm | RMSNorm |
|---|---|---|
| Reductions | 2 (mean, variance) | **1 (mean of squares)** |
| Parameters | gamma + beta | **gamma only** |
| Mean subtraction | Yes | **No** |
| HBM passes (naive) | 4+ | 3 |
| Triton passes | 1 | 1 |
| Used by | BERT, GPT-2 | **LLaMA, Mistral, Gemma, Qwen, DeepSeek** |

RMSNorm was proposed by Zhang & Sennrich (2019). The insight: the mean subtraction in LayerNorm provides minimal benefit for the extra compute cost. Modern LLMs universally adopted RMSNorm because it's simpler, faster, and empirically equivalent.

## Why This Is What Labs Ship First

Fused normalization kernels are the **most practical** custom kernel for several reasons:

1. **High frequency** — every transformer block calls normalization 1-2 times. A 32-layer model calls it 64 times per forward pass.
2. **Large fusion benefit** — 4+ HBM passes → 1 pass. Consistent, guaranteed speedup.
3. **Simple kernel** — same 1D grid structure as softmax. No complex tiling like matmul.
4. **Unlike matmul, PyTorch's built-in isn't already optimal** — `torch.matmul` calls highly-optimized cuBLAS. But `torch.layer_norm` is a composition of separate CUDA kernels, leaving room for fusion.

This is literally what NVIDIA Apex, xFormers, and flash-attn ship as their normalization primitives.

## What to Notice

1. **Triton matches `torch.layer_norm` closely** — max difference should be near 1e-6, similar to softmax. Both compute the same reductions in the same order.

2. **Fusion wins for the same reason as softmax** — the speedup comes from eliminating redundant HBM reads, not faster arithmetic. LayerNorm fuses 4+ passes into 1, even more than softmax's 3-to-1.

3. **RMSNorm is faster than LayerNorm** — fewer operations (1 reduction vs 2, no bias). This matches the trend in modern LLMs away from full LayerNorm.

4. **First kernel with learnable parameters** — gamma and beta are loaded from separate pointers. This is how all production kernels work: the kernel receives weight pointers alongside input pointers.

5. **`0.0` padding, not `-inf`** — unlike softmax (where `-inf` ensures `exp(-inf) = 0`), LayerNorm pads with `0.0` because zeros don't affect mean/variance when we divide by `n_cols` (the true column count), not `BLOCK_SIZE`.

6. **Same grid as softmax** — both use 1D grids with one program per row. The difference is what happens inside: softmax does max/exp/sum/div; LayerNorm does mean/var/normalize/scale.

## Resources

- [Triton Layer Norm Tutorial](https://triton-lang.org/main/getting-started/tutorials/05-layer-norm.html) — Official Triton tutorial
- [Root Mean Square Layer Normalization (Zhang & Sennrich, 2019)](https://arxiv.org/abs/1910.07467) — The RMSNorm paper
- [GPU MODE Lectures](https://github.com/gpu-mode/lectures) — Community GPU programming course
- [NVIDIA Apex Fused LayerNorm](https://github.com/NVIDIA/apex) — Production fused kernel implementation